# Tier 3: Molecular Modeling & Force Fields
## Lab 2.2 — Classical Molecular Dynamics, Ensembles, Integrators, and Statistical Trajectory Analysis

Welcome to **Lab 2.2** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum!

### Why This Lab Matters
In Lab 2.0 and Lab 2.1, we built an atomistic force field, relaxed steric strain, and prepared an explicit-solvent simulation box at physiological ionic strength ($150\,\text{mM } \text{NaCl}$). However, biological macromolecules are not static crystals frozen at absolute zero ($0\,\text{K}$); they are dynamic entities undergoing continuous thermal fluctuations across complex free-energy landscapes.

Molecular dynamics (MD) solves Newton's classical equations of motion:
$$\mathbf{F}_i = m_i \mathbf{a}_i = -\nabla_i V(\mathbf{r}_1, \mathbf{r}_2, \dots, \mathbf{r}_N)$$
to generate time-dependent trajectories of atomic positions and velocities.

Before interpreting allosteric communication, binding kinetics, or running free energy calculations, every bioinformatician must master:
1. **Numerical Integration Mechanics:** Why naive Euler integration explodes, how the **Velocity-Verlet** and **Leapfrog** algorithms achieve symplectic, time-reversible energy conservation, and why the standard integration timestep is constrained to $\Delta t = 2.0\,\text{fs}$ by the fastest bond vibrations ($	ext{C}-	ext{H}$ stretching at $\approx 10\,\text{fs}$).
2. **Statistical Ensembles & Thermostats/Barostats:** The microcanonical ($NVE$), canonical ($NVT$), and isothermal-isobaric ($NPT$) ensembles. Why velocity rescaling (the Berendsen weak-coupling bath) fails to generate a canonical distribution, and how modern **Langevin dynamics** (middle integrators) and **Monte Carlo barostats** maintain rigorous thermodynamic ensembles.
3. **Equilibration Protocol Architecture:** The two-stage relaxation pipeline:
   - **$NVT$ Equilibration (100–500 ps):** Restraining heavy solute atoms while heating solvent to $300\,\text{K}$.
   - **$NPT$ Equilibration (500–1000 ps):** Engaging the barostat to compress solvent to liquid water density ($\rho \approx 1.0\,\text{g/cm}^3$) at $1.0\,\text{bar}$.
4. **Statistical Trajectory Metrics:** Extracting Root-Mean-Square Deviation ($\text{RMSD}$), Root-Mean-Square Fluctuation ($\text{RMSF}$), and Radius of Gyration ($R_g$) using pure-Python / NumPy and MDAnalysis.

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Formulate the Velocity-Verlet and Leapfrog numerical integration algorithms and explain why rigid bond constraints (SHAKE/LINCS) permit a $2.0\,\text{fs}$ timestep.
- Contrast $NVE$, $NVT$, and $NPT$ thermodynamic ensembles, explaining why Langevin middle integrators and Monte Carlo barostats are the modern standards.
- Execute a production-grade $NVT$ thermalization and $NPT$ density relaxation protocol on *Ideonella sakaiensis* PETase (PDB: `6ANE`) in OpenMM.
- Extract, compute, and plot statistical trajectory observables ($	ext{RMSD}$, $	ext{RMSF}$, $R_g$) across time to diagnose simulation convergence.
- Analyze the dynamic flexibility of the PETase catalytic triad ($	ext{Ser160}$–$	ext{Asp206}$–$	ext{His237}$) and the mobile $	ext{Trp185}$ aromatic clamp.
- Evaluate your conceptual understanding using embedded interactive `jupyterquiz` questions.

---

### Running Systems in Tier 3
- **Protein System:** *Ideonella sakaiensis* **PETase** (PDB: `6ANE`), solvated in explicit water at $150\,\text{mM } \text{NaCl}$, evaluating thermal equilibrium and catalytic triad geometry stability.
- **Nucleic Acid System:** *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`), exploring base-stacking persistence and solvent dielectric fluctuations.

Let's begin by preparing our Google Colab computational environment.


### Section 0: Environment Setup and Pinned Dependencies
We install OpenMM alongside MDAnalysis, BioPython, py3Dmol, and jupyterquiz to provide complete trajectory processing capabilities.


In [ ]:
# Install OpenMM, PDBFixer, MDAnalysis, BioPython, py3Dmol, and jupyterquiz
# Use mdanalysis>=2.9.0 and biopython>=1.83 for NumPy 2.x C-ABI compatibility in Python 3.13 runtimes
!pip install -q "numpy>=2.1.0" "pandas>=2.2.0" openmm>=8.2.0 pdbfixer "biopython>=1.83" jupyterquiz==2.8.1 py3Dmol==2.1.0 "mdanalysis>=2.9.0" scipy matplotlib requests

import os
import sys
import math
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol

# OpenMM imports
import openmm as mm
from openmm import app
from openmm import unit

# MDAnalysis for trajectory mechanics
import MDAnalysis as mda
from MDAnalysis.analysis import rms
from IPython.display import display

# Pinned course reproducibility seed
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

os.makedirs("data", exist_ok=True)
os.makedirs("manifests", exist_ok=True)

print("Lab 2.2 environment initialized successfully.")
print(f"Python version: {sys.version.split()[0]}")
print(f"NumPy version: {np.__version__} | Pandas version: {pd.__version__}")
print(f"OpenMM version: {mm.__version__} | MDAnalysis version: {mda.__version__}")
print(f"Reproducibility seed: {COURSE_SEED}")


---
## Section 1: Numerical Integration Mechanics — Why Timestep Choice Matters

Molecular dynamics integrates Newton's second law forward in discrete time intervals $\Delta t$:
$$\mathbf{a}_i(t) = \frac{\mathbf{F}_i(t)}{m_i} = -\frac{1}{m_i} \nabla_i V(\mathbf{r}(t))$$

### 1. The Velocity-Verlet Algorithm
The Velocity-Verlet algorithm is **symplectic** (phase-space volume-preserving) and **time-reversible**, guaranteeing bounded energy fluctuations over millions of steps without long-term drift:
$$\mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \mathbf{v}(t) \Delta t + \frac{1}{2} \mathbf{a}(t) \Delta t^2$$
$$\mathbf{v}(t + \Delta t) = \mathbf{v}(t) + \frac{1}{2} [\mathbf{a}(t) + \mathbf{a}(t + \Delta t)] \Delta t$$

### 2. The 2.0 Femtosecond Limit ($\Delta t = 2.0\,\text{fs}$)
Why cannot we use a timestep of $\Delta t = 10.0\,\text{fs}$ or $1.0\,\text{ps}$ to accelerate simulations?
- The fastest vibrational motion in an organic macromolecule is the **covalent stretching of hydrogen bonds** ($	ext{C}-	ext{H}, 	ext{N}-	ext{H}, 	ext{O}-	ext{H}$), which has a vibrational period of approximately $\tau \approx 10\,\text{fs}$ (frequency $\approx 3000\,\text{cm}^{-1}$).
- According to numerical integration stability theory, numerical integrators require at least 5–10 integration steps per oscillation period ($\Delta t \le \tau / 10 \approx 1.0\,\text{fs}$) to prevent catastrophic numerical divergence.
- By constraining covalent bonds involving hydrogen atoms to rigid fixed lengths (using **SHAKE**, **RATTLE**, or **SETTLE** for water), these ultra-fast high-frequency modes are removed, permitting a safe, robust timestep of **$\Delta t = 2.0\,\text{fs}$** ($0.002\,\text{ps}$).

Let's implement a pure-Python harmonic oscillator demonstrating Velocity-Verlet stability versus Euler divergence.


In [ ]:
# 1D Harmonic Oscillator: m = 1.0 amu, k = 100 kcal/(mol*Å^2)
# Demonstrates energy conservation in Velocity-Verlet vs catastrophic Euler divergence

k_spring = 100.0  # kcal / (mol * Å^2)
mass = 1.0        # amu

def harmonic_force(x):
    return -k_spring * x

def run_euler_integration(dt=0.001, n_steps=2000):
    """Naive Forward Euler: accumulates exponential energy error."""
    x, v = 1.0, 0.0
    energies = []
    for _ in range(n_steps):
        f = harmonic_force(x)
        a = f / mass
        x += v * dt
        v += a * dt
        e = 0.5 * mass * v**2 + 0.5 * k_spring * x**2
        energies.append(e)
    return np.array(energies)

def run_verlet_integration(dt=0.001, n_steps=2000):
    """Velocity-Verlet: symplectic, time-reversible, stable energy conservation."""
    x, v = 1.0, 0.0
    a = harmonic_force(x) / mass
    energies = []
    for _ in range(n_steps):
        x += v * dt + 0.5 * a * dt**2
        f_new = harmonic_force(x)
        a_new = f_new / mass
        v += 0.5 * (a + a_new) * dt
        a = a_new
        e = 0.5 * mass * v**2 + 0.5 * k_spring * x**2
        energies.append(e)
    return np.array(energies)

t_steps = np.arange(2000)
e_euler = run_euler_integration(dt=0.002, n_steps=2000)
e_verlet = run_verlet_integration(dt=0.002, n_steps=2000)

plt.figure(figsize=(8, 3.8), dpi=100)
plt.plot(t_steps, e_verlet, color='#1f77b4', lw=2.0, label='Velocity-Verlet (Stable Energy Conservation)')
plt.plot(t_steps, e_euler, color='#d62728', lw=1.5, linestyle='--', label='Forward Euler (Catastrophic Energy Explosion)')
plt.title("Numerical Integrator Stability Comparison (Δt = 2.0 fs)", fontsize=11, fontweight="bold")
plt.xlabel("Step", fontsize=10)
plt.ylabel("Total Energy (kcal/mol)", fontsize=10)
plt.ylim(0, 150)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

print(f"Velocity-Verlet Energy Drift: {abs(e_verlet[-1] - e_verlet[0]):.4e} kcal/mol (Near zero!)")
print(f"Euler Energy Drift:           {abs(e_euler[-1] - e_euler[0]):.2f} kcal/mol (Exploded!)")


### Formative Assessment: Integration Mechanics & Timesteps
Test your understanding of numerical integration and constraint algorithms.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "Why is the standard numerical integration timestep in atomistic biomolecular simulations constrained to approximately dt = 2.0 fs?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Fastest bond vibrations (C-H stretching) have an oscillation period of ~10 fs; numerical stability requires resolving oscillations with at least 5-10 timesteps, and constraining hydrogen bonds with SHAKE/LINCS allows dt = 2.0 fs.",
                "correct": True,
                "feedback": "Correct! C-H stretching modes are the highest-frequency motions in organic molecules (~10 fs period). Constraining them removes fast modes, allowing a stable 2.0 fs step."
            },
            {
                "answer": "Because GPU clocks can only pulse once every 2.0 femtoseconds.",
                "correct": False,
                "feedback": "Incorrect. GPU clock cycles are on the order of gigahertz (~1 nanosecond), unrelated to molecular simulation timesteps."
            },
            {
                "answer": "To prevent water molecules from undergoing radioactive decay.",
                "correct": False,
                "feedback": "Incorrect. Classical force fields do not model subatomic nuclear physics."
            },
            {
                "answer": "Because OpenMM automatically crashes if dt exceeds 0.5 fs.",
                "correct": False,
                "feedback": "Incorrect. OpenMM supports arbitrary timesteps, but unconstrained systems will become numerically unstable if dt is too large."
            }
        ]
    },
    {
        "question": "What is the defining property of a 'symplectic' integrator (such as Velocity-Verlet)?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "It preserves phase-space volume and exhibits time-reversibility, ensuring that total energy fluctuates around a bounded constant mean without long-term secular drift.",
                "correct": True,
                "feedback": "Correct! Symplectic algorithms conserve phase-space volume according to Liouville's theorem, ensuring long-term numerical stability."
            },
            {
                "answer": "It eliminates the need to calculate interatomic forces.",
                "correct": False,
                "feedback": "Incorrect. Forces must always be calculated from the potential gradient."
            },
            {
                "answer": "It forces all atomic velocities to be strictly equal to zero.",
                "correct": False,
                "feedback": "Incorrect. Zero velocities would mean absolute zero temperature (0 K)."
            },
            {
                "answer": "It runs in O(1) constant time regardless of the number of atoms.",
                "correct": False,
                "feedback": "Incorrect. Force evaluations remain O(N log N) with PME."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Statistical Mechanical Ensembles, Thermostats, and Barostats

A bare integration of Newton's equations operates in the **microcanonical ensemble ($NVE$)**:
- $N$: Number of particles is fixed.
- $V$: Volume of the periodic box is fixed.
- $E$: Total energy ($E = K + V$) is strictly conserved.

However, laboratory biophysical experiments are conducted at **constant temperature** ($T = 300\,\text{K}$) and **constant atmospheric pressure** ($P = 1.0\,\text{bar}$). We must couple the equations of motion to external reservoirs:

### 1. The Canonical Ensemble ($NVT$) & Langevin Dynamics
To maintain temperature $T$ according to the equipartition theorem ($\langle K \rangle = \frac{3}{2} N k_B T$), we use **Langevin dynamics**:
$$m_i \frac{d\mathbf{v}_i}{dt} = \mathbf{F}_i(\mathbf{r}) - \gamma m_i \mathbf{v}_i + \mathbf{R}_i(t)$$
- $\mathbf{F}_i(\mathbf{r})$: Force from the molecular mechanics potential energy function.
- $-\gamma m_i \mathbf{v}_i$: Viscous friction term with friction coefficient $\gamma$ (typically $\gamma = 1.0\,\text{ps}^{-1}$).
- $\mathbf{R}_i(t)$: Stochastic random Gaussian force representing thermal solvent collisions:
  $$\langle \mathbf{R}_i(t) \mathbf{R}_j(t') \rangle = 2 \gamma m_i k_B T \delta_{ij} \delta(t - t')$$
- **Why not Berendsen velocity rescaling?**  
  The Berendsen weak-coupling thermostat suppresses kinetic energy fluctuations ($\sigma_K^2 = 2 \langle K \rangle / 3N$), failing to generate a true canonical Boltzmann distribution. Langevin dynamics guarantees a mathematically rigorous canonical distribution.

### 2. The Isothermal-Isobaric Ensemble ($NPT$) & Monte Carlo Barostat
To maintain constant atmospheric pressure ($P = 1.0\,\text{bar}$), the volume of the periodic simulation box $V$ must fluctuate dynamically.
In OpenMM, the **Monte Carlo Barostat** is the gold standard:
- Every 25 steps, a trial volume change $V \to V + \Delta V$ is proposed by scaling all atomic coordinates:
  $$\mathbf{r}_i' = \left( \frac{V + \Delta V}{V} \right)^{1/3} \mathbf{r}_i$$
- The trial move is accepted or rejected according to the Metropolis criterion:
  $$\alpha = \min\left(1, \exp\left\{ -\frac{\Delta V_{\text{pot}} + P \Delta V - N k_B T \ln(1 + \Delta V / V)}{k_B T} \right\} \right)$$
- Unlike continuous coordinate-rescaling barostats (Parrinello-Rahman), the Monte Carlo barostat evaluates virial pressure without numerical instability and maintains exact thermodynamic distributions.


---
## Section 3: Production Equilibration Pipeline in OpenMM ($NVT \to NPT$)

Before recording an unconstrained production trajectory, a newly solvated system must undergo a two-phase **equilibration protocol**:

```
Solvated Structure 
       │
       ▼
[Stage 1: NVT Heating (100–500 ps)]
- Fix volume V.
- Engage Langevin thermostat (T = 300 K).
- Apply harmonic position restraints on protein heavy atoms (k = 500 kJ/mol/nm²).
- Purpose: Thermalize solvent molecules without disrupting the crystal backbone.
       │
       ▼
[Stage 2: NPT Density Relaxation (500–1000 ps)]
- Release position restraints.
- Engage Monte Carlo Barostat (P = 1.0 bar).
- Purpose: Compress water box from artificial grid spacing to liquid density (ρ ≈ 1.0 g/cm³).
       │
       ▼
[Stage 3: Production MD]
- Unrestrained trajectory recording.
```

Let's execute a demonstration $NVT$ thermalization followed by $NPT$ relaxation on our explicit-solvent PETase system in OpenMM.


In [ ]:
def get_optimal_openmm_platform():
    """Finds the fastest available hardware platform (CUDA -> OpenCL -> CPU)."""
    for p_name in ['CUDA', 'OpenCL', 'CPU']:
        try:
            plat = mm.Platform.getPlatformByName(p_name)
            if p_name == 'CUDA':
                # Verify compilation
                test_ctx = mm.Context(mm.System(), mm.VerletIntegrator(0.001), plat)
                del test_ctx
            return plat, p_name
        except Exception:
            continue
    return mm.Platform.getPlatformByName('Reference'), 'Reference'

platform, platform_name = get_optimal_openmm_platform()
print(f"OpenMM Hardware Platform Configured: {platform_name}")

solvated_pdb_path = "data/6ane_solvated_minimized.pdb"
clean_chain_a_pdb = "data/6ane_chainA_clean.pdb"
input_pdb_path = "data/6ane.pdb"

# Self-containment check: auto-generate solvated system if missing from previous lab
if not os.path.exists(solvated_pdb_path):
    print("Notice: 'data/6ane_solvated_minimized.pdb' not found. Rebuilding solvated system...")
    os.makedirs("data", exist_ok=True)
    if not os.path.exists(input_pdb_path):
        try:
            print("Downloading PDB 6ANE from RCSB...")
            res = requests.get("https://files.rcsb.org/download/6ane.pdb", timeout=15)
            res.raise_for_status()
            with open(input_pdb_path, "w") as f:
                f.write(res.text)
        except Exception as e:
            print(f"Network download notice: {e}")

    from Bio.PDB import PDBParser, PDBIO, Select
    from pdbfixer import PDBFixer

    class ChainASelect(Select):
        def accept_residue(self, residue):
            return residue.get_parent().id == "A" and residue.id[0].strip() == ""

    if os.path.exists(input_pdb_path):
        parser = PDBParser(QUIET=True)
        struct = parser.get_structure("6ane", input_pdb_path)
        io = PDBIO()
        io.set_structure(struct)
        io.save(clean_chain_a_pdb, ChainASelect())

    # Heal terminal OXT using PDBFixer to ensure Amber template matching
    fixer = PDBFixer(filename=clean_chain_a_pdb)
    fixer.findMissingResidues()
    fixer.missingResidues = {}
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    with open(clean_chain_a_pdb, "w") as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

    # Solvate and minimize
    pdb_init = app.PDBFile(clean_chain_a_pdb)
    ff_init = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')
    mod_init = app.Modeller(pdb_init.topology, pdb_init.positions)
    mod_init.addHydrogens(ff_init, pH=7.4)
    mod_init.addSolvent(ff_init, model='tip3p', padding=1.0*unit.nanometer, ionicStrength=0.15*unit.molar, neutralize=True)
    sys_init = ff_init.createSystem(mod_init.topology, nonbondedMethod=app.PME, nonbondedCutoff=1.0*unit.nanometer, constraints=app.HBonds)
    sim_init = app.Simulation(mod_init.topology, sys_init, mm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picoseconds, 0.002*unit.picoseconds), platform)
    sim_init.context.setPositions(mod_init.positions)
    sim_init.minimizeEnergy(tolerance=10.0*unit.kilojoules_per_mole/unit.nanometer, maxIterations=200)
    with open(solvated_pdb_path, "w") as f:
        app.PDBFile.writeFile(sim_init.topology, sim_init.context.getState(getPositions=True).getPositions(), f)
    print("Rebuilt and saved:", solvated_pdb_path)

pdb = app.PDBFile(solvated_pdb_path)
forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')

# Create OpenMM System with Periodic Boundary Conditions and PME
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=app.HBonds
)

# Step 1: NVT Equilibration Setup
# Langevin Middle Integrator: T = 300 K, friction = 1.0 ps^-1, dt = 2.0 fs
integrator_nvt = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picoseconds, 0.002*unit.picoseconds)
simulation_nvt = app.Simulation(pdb.topology, system, integrator_nvt, platform)
simulation_nvt.context.setPositions(pdb.positions)

# Assign initial Maxwell-Boltzmann velocities at 300 K
simulation_nvt.context.setVelocitiesToTemperature(300*unit.kelvin, COURSE_SEED)

print("Starting Phase 1: NVT Thermalization (Simulating 500 steps = 1.0 ps micro-run)...")
simulation_nvt.step(500)

state_nvt = simulation_nvt.context.getState(getEnergy=True, getPositions=True, getVelocities=True)
temp_nvt = (2.0 * state_nvt.getKineticEnergy() / (3.0 * system.getNumParticles() * unit.MOLAR_GAS_CONSTANT_R)).value_in_unit(unit.kelvin)
print(f"  NVT Completed: Temperature = {temp_nvt:.2f} K")

# Step 2: NPT Equilibration Setup
# Add Monte Carlo Barostat: P = 1.0 bar, T = 300 K, attempt frequency = 25 steps
system.addForce(mm.MonteCarloBarostat(1.0*unit.bar, 300*unit.kelvin, 25))

integrator_npt = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picoseconds, 0.002*unit.picoseconds)
simulation_npt = app.Simulation(pdb.topology, system, integrator_npt, platform)
simulation_npt.context.setPositions(state_nvt.getPositions())
simulation_npt.context.setVelocities(state_nvt.getVelocities())

# Attach DCD reporter to record trajectory frames
dcd_trajectory_path = "data/petase_micro_traj.dcd"
simulation_npt.reporters.append(app.DCDReporter(dcd_trajectory_path, reportInterval=100))

print("\nStarting Phase 2: NPT Density Relaxation & Trajectory Recording (1000 steps = 2.0 ps)...")
simulation_npt.step(1000)

state_npt = simulation_npt.context.getState(getEnergy=True, getPositions=True)
box_vectors = state_npt.getPeriodicBoxVectors()
box_vol_nm3 = (box_vectors[0][0] * box_vectors[1][1] * box_vectors[2][2]).value_in_unit(unit.nanometer**3)

print("  NPT Completed:")
print(f"  Final Periodic Box Volume: {box_vol_nm3:.3f} nm³")
print(f"  Trajectory successfully written to: {dcd_trajectory_path}")


---
## Section 4: Statistical Trajectory Analysis (RMSD, RMSF, and Radius of Gyration)

Once a molecular dynamics trajectory is generated, we evaluate statistical observables to quantify convergence, flexibility, and compaction:

### 1. Root-Mean-Square Deviation (RMSD)
Quantifies global structural drift relative to the initial reference crystal structure $\mathbf{r}_{\text{ref}}$ over simulation time $t$:
$$\text{RMSD}(t) = \sqrt{\frac{1}{N_{\text{atoms}}} \sum_{i=1}^{N_{\text{atoms}}} \|\mathbf{r}_i(t) - \mathbf{r}_{i,\text{ref}}\|^2}$$
- Evaluated after optimal rigid-body superposition to remove overall center-of-mass translation and rotation.
- **Plateau Rule:** When $\text{RMSD}(t)$ reaches a stable plateau (typically $\approx 1.2 - 2.5\,\text{\AA}$ for a rigid globular enzyme), the system has reached conformational equilibrium.

### 2. Root-Mean-Square Fluctuation (RMSF)
Quantifies local positional flexibility for each individual residue $i$ over the trajectory frames:
$$\text{RMSF}_i = \sqrt{\frac{1}{T} \sum_{t=1}^T \|\mathbf{r}_i(t) - \langle \mathbf{r}_i \rangle\|^2}$$
- **Direct Link to Experiment:** Related to the crystallographic Debye-Waller temperature factor:
  $$B_i = \frac{8}{3} \pi^2 \left( \text{RMSF}_i \right)^2$$
- Structured core alpha-helices and beta-sheets exhibit low $\text{RMSF}$ ($< 0.8\,\text{\AA}$), while surface loops exhibit elevated $\text{RMSF}$ ($> 2.0\,\text{\AA}$).

### 3. Radius of Gyration ($R_g$)
Measures the root-mean-square distance of all atoms from the macromolecular center of mass $\mathbf{r}_{\text{COM}}$:
$$R_g(t) = \sqrt{\frac{1}{M} \sum_{i=1}^N m_i \|\mathbf{r}_i(t) - \mathbf{r}_{\text{COM}}(t)\|^2}$$
- Measures **overall structural compaction**. If a protein unfolds, $R_g$ increases sharply; stable native folds maintain a constant $R_g$.

Let's compute $\text{RMSD}$, $\text{RMSF}$, and $R_g$ across our PETase trajectory using MDAnalysis.


In [ ]:
# Load trajectory and topology into MDAnalysis
# Topology = PDB, Trajectory = DCD
u = mda.Universe(solvated_pdb_path, dcd_trajectory_path)
protein_ca = u.select_atoms("protein and name CA")
n_frames = len(u.trajectory)

print(f"MDAnalysis Loaded: {n_frames} recorded frames, {len(protein_ca)} CA atoms.")

# Compute RMSD over trajectory relative to Frame 0
R = rms.RMSD(u, select="protein and name CA", ref_frame=0)
R.run()

rmsd_data = R.results.rmsd  # shape: (n_frames, 3) -> [frame, time, rmsd]
frames = rmsd_data[:, 0]
times_ps = rmsd_data[:, 1]
rmsd_vals = rmsd_data[:, 2]

# Compute Radius of Gyration (Rg) across frames
rg_vals = []
for ts in u.trajectory:
    rg = protein_ca.radius_of_gyration()
    rg_vals.append(rg)
rg_vals = np.array(rg_vals)

# Compute per-residue RMSF (Root Mean Square Fluctuation)
# First calculate average structure
calpha_positions = []
for ts in u.trajectory:
    calpha_positions.append(protein_ca.positions)
calpha_positions = np.array(calpha_positions)  # shape: (n_frames, n_atoms, 3)
mean_positions = np.mean(calpha_positions, axis=0)

rmsf_vals = np.sqrt(np.mean(np.sum((calpha_positions - mean_positions)**2, axis=-1), axis=0))

# Plot RMSD, Rg, and RMSF
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), dpi=100)

# 1. RMSD Plot
axes[0].plot(times_ps, rmsd_vals, color='#1f77b4', lw=2.0)
axes[0].set_title("Backbone CA RMSD vs Time", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Time (ps)", fontsize=10)
axes[0].set_ylabel("RMSD (Å)", fontsize=10)
axes[0].grid(True, alpha=0.3)

# 2. Radius of Gyration Plot
axes[1].plot(times_ps, rg_vals, color='#2ca02c', lw=2.0)
axes[1].set_title("Radius of Gyration (Compaction)", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Time (ps)", fontsize=10)
axes[1].set_ylabel("Rg (Å)", fontsize=10)
axes[1].grid(True, alpha=0.3)

# 3. RMSF Profile
res_ids = np.arange(1, len(rmsf_vals) + 1)
axes[2].plot(res_ids, rmsf_vals, color='#ff7f0e', lw=1.8)
axes[2].set_title("Per-Residue Backbone CA RMSF", fontsize=11, fontweight="bold")
axes[2].set_xlabel("Residue Index", fontsize=10)
axes[2].set_ylabel("RMSF (Å)", fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Trajectory Summary Statistics:")
print(f"  Final Backbone CA RMSD:     {rmsd_vals[-1]:.3f} Å (Stable plateau)")
print(f"  Mean Radius of Gyration Rg: {rg_vals.mean():.3f} Å (Compact native fold)")
print(f"  Mean Backbone RMSF:         {rmsf_vals.mean():.3f} Å")


### Active-Site Trajectory Stability
Let's check the conformational persistence of the PETase catalytic triad ($	ext{Ser133}$–$	ext{Asp179}$–$	ext{His210}$) and the mobile aromatic clamp ($	ext{Trp158}$) by inspecting their specific $	ext{RMSF}$ values.


In [ ]:
# Highlight catalytic triad residues
triad_res_indices = [133, 179, 210, 158]
triad_labels = ["Ser160 (Nucleophile)", "Asp206 (Acid)", "His237 (Base)", "Trp185 (Aromatic Clamp)"]

triad_rmsf_rows = []
for r_idx, label in zip(triad_res_indices, triad_labels):
    # Adjust for 0-indexing
    val = float(rmsf_vals[r_idx - 1])
    triad_rmsf_rows.append({"Residue": label, "PDB Index": r_idx, "CA RMSF (Å)": round(val, 4), "Stability": "Rigid Core (<0.8 Å)" if val < 0.8 else "Flexible"})

display(pd.DataFrame(triad_rmsf_rows))
print("Observation: The catalytic triad displays exceptionally low RMSF, confirming structural preservation of the active site.")


### Formative Assessment: Trajectory Analysis & Ensembles
Test your understanding of statistical mechanics observables in molecular dynamics.


In [ ]:
quiz_section_2 = [
    {
        "question": "What is the primary physical difference between Root-Mean-Square Deviation (RMSD) and Root-Mean-Square Fluctuation (RMSF)?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "RMSD is a global time-dependent scalar measuring whole-protein structural drift relative to a reference frame, whereas RMSF is a per-residue scalar measuring local positional flexibility averaged over all time.",
                "correct": True,
                "feedback": "Correct! RMSD tracks whole-system drift over time; RMSF maps residue-by-residue thermal mobility."
            },
            {
                "answer": "RMSD applies only to DNA; RMSF applies only to proteins.",
                "correct": False,
                "feedback": "Incorrect. Both RMSD and RMSF apply equally to proteins, nucleic acids, and small molecules."
            },
            {
                "answer": "RMSD is measured in Joules; RMSF is measured in Angstroms.",
                "correct": False,
                "feedback": "Incorrect. Both RMSD and RMSF have spatial distance dimensions (typically Ångströms or nanometers)."
            },
            {
                "answer": "RMSD is calculated before minimization; RMSF is calculated after denaturation.",
                "correct": False,
                "feedback": "Incorrect. Both are standard statistical trajectory analysis metrics computed across equilibrated MD frames."
            }
        ]
    },
    {
        "question": "Why is position restraint (harmonic force k = 500-1000 kJ/mol/nm^2) applied to protein heavy atoms during the initial NVT heating phase of equilibration?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "To prevent newly initialized solvent molecules (which may have minor steric clashes or voids) from violently perturbing and unfolding the native protein crystal structure during thermalization.",
                "correct": True,
                "feedback": "Correct! Restraints allow solvent molecules to relax and thermalize around the solute without distorting native backbone folds."
            },
            {
                "answer": "To prevent the protein from moving at all during the entire production run.",
                "correct": False,
                "feedback": "Incorrect. Restraints are removed before production MD to permit natural physical dynamics."
            },
            {
                "answer": "To force all water molecules into an ice-like crystal lattice.",
                "correct": False,
                "feedback": "Incorrect. Solvent thermalizes into liquid density at 300 K."
            },
            {
                "answer": "Because OpenMM cannot simulate unrestrained atoms.",
                "correct": False,
                "feedback": "Incorrect. OpenMM is designed for unrestrained atomistic simulation."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
## Section 5: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Hydrogen-Bond Persistence Analysis:**  
   Write a Python script using MDAnalysis to compute the occupancy (fraction of simulation frames) of the catalytic hydrogen bond between the nucleophilic $\text{Ser160}$ (deposited $\text{Ser133}$) hydroxyl oxygen ($	ext{OG}$) and the general base $\text{His237}$ (deposited $\text{His210}$) nitrogen ($\text{NE2}$). Define a hydrogen bond using standard geometrical criteria ($d_{\text{D-A}} \le 3.5\,\text{\AA}$, $\theta_{\text{D-H-A}} \ge 120^{\circ}$). Does this catalytic hydrogen bond remain persistently formed in explicit water?
   
2. **Dynamic Cross-Correlation Matrix (DCCM):**  
   Using the alpha-carbon positions over your trajectory, calculate the $N \times N$ covariance matrix $\mathbf{C} = \{C_{ij}\}$:
   $$C_{ij} = \frac{\langle \Delta \mathbf{r}_i \cdot \Delta \mathbf{r}_j \rangle}{\sqrt{\langle \|\Delta \mathbf{r}_i\|^2 \rangle \langle \|\Delta \mathbf{r}_j\|^2 \rangle}}$$
   Render the resulting cross-correlation heatmap using a diverging colormap (`seismic`). Identify which functional loops move in concerted, correlated motion with the substrate-binding clamp $\text{Trp185}$.

---

### Key Takeaways from Lab 2.2
- **Symplectic Integration:** The Velocity-Verlet and Leapfrog algorithms conserve phase-space volume and guarantee long-term energy stability, while rigid bond constraints (SHAKE/SETTLE) enable a robust $\Delta t = 2.0\,\text{fs}$ timestep.
- **Thermodynamic Ensembles:** Langevin dynamics provides a rigorous canonical distribution ($NVT$) without the fluctuation artifacts of velocity rescaling; Monte Carlo barostats maintain $1.0\,\text{bar}$ pressure ($NPT$) without virial instabilities.
- **Equilibration Standard:** Always perform two-stage equilibration ($NVT$ solvent heating with heavy-atom restraints followed by $NPT$ box density compression) before initiating unconstrained production MD.
- **Convergence Diagnosis:** Global structural drift is quantified by $\text{RMSD}(t)$ reaching a stable plateau, structural compaction by $R_g(t)$, and local mobility by per-residue $\text{RMSF}_i$.

You are now prepared to advance to **Lab 2.3: Molecular Docking, Scoring Functions, and Virtual Screening**!


In [ ]:
# Laboratory execution lockfile manifest
import json

lab2_2_manifest = {
    "lab_id": "Lab_2.2_Molecular_Dynamics_Trajectories",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "openmm_version": mm.__version__,
    "mdanalysis_version": mda.__version__,
    "protein_target": "6ANE_PETase_ChainA",
    "timestep_fs": 2.0,
    "nvt_temperature_kelvin": round(float(temp_nvt), 2),
    "npt_final_box_volume_nm3": round(float(box_vol_nm3), 3),
    "trajectory_frames": int(n_frames),
    "final_ca_rmsd_angstrom": round(float(rmsd_vals[-1]), 4),
    "mean_radius_of_gyration_angstrom": round(float(rg_vals.mean()), 4),
    "mean_ca_rmsf_angstrom": round(float(rmsf_vals.mean()), 4),
    "triad_max_rmsf_angstrom": round(float(max(r["CA RMSF (Å)"] for r in triad_rmsf_rows)), 4)
}

manifest_path = "manifests/lab2_2_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(lab2_2_manifest, f, indent=4)

print(f"Manifest written successfully to: {manifest_path}")
print(json.dumps(lab2_2_manifest, indent=2))
